# SLTFM · Evaluation: base Llama 3 8B vs. SLTFM adapter (checkpoint-1100)

Compares **plain Llama 3 8B** with **Llama 3 8B + the SLTFM continual-pretraining adapter** on Sri Lankan Tamil.

1. **Held-out text (measured):** bits per character on validation books the model never trained on. Lower = the model predicts Sri Lankan Tamil text better.
2. **Example questions:** Sri Lankan Tamil questions (general, corpus-grounded and dialect), answered by both models.
3. **Text continuation:** both models continue Sri Lankan Tamil prompts.

Results are saved to `/kaggle/working/sltfm_eval_results.md` (or the current folder on Colab) for sharing.

**Setup:** one GPU is enough (Kaggle *GPU T4* or Colab *T4*), Internet on, and a secret named **`HF_TOKEN`** (the ctnlpr token, read access is enough). Run All takes about 30–40 minutes.

Note: this is a *base* model (not instruction-tuned), so questions use a one-example prompt format and short answers are expected.

In [ ]:
!pip install -q -U "transformers>=4.51,<4.57" "peft>=0.15,<0.18" "accelerate>=1.6,<2" "bitsandbytes>=0.45" "datasets>=3.2"
!nvidia-smi -L

In [ ]:
# ============================ settings ============================
BASE_MODEL  = "meta-llama/Meta-Llama-3-8B"
ADAPTER     = "ctnlpr/Llama-3-SLTFM-8B-CPT-LoRA"
SUBFOLDER   = "checkpoint-1100"
DATASET_ID  = "ctnlpr/sri-lankan-tamil-corpus"
EVAL_TOKENS = 60_000      # held-out tokens scored per model (~45k Tamil characters)
WINDOW      = 2048
# =================================================================
import os
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    OUT_DIR = "/kaggle/working"
except Exception:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = os.environ.get("HF_TOKEN")
    OUT_DIR = "."
assert token, "Add a secret named HF_TOKEN"
os.environ["HF_TOKEN"] = token
print("output folder:", OUT_DIR)

In [ ]:
import math, time, json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

tok = AutoTokenizer.from_pretrained(BASE_MODEL, token=token)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                         bnb_4bit_compute_dtype=torch.float16)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, torch_dtype=torch.float16,
                                            device_map={"": 0}, token=token)
model = PeftModel.from_pretrained(base, ADAPTER, subfolder=SUBFOLDER, token=token)
model.eval()
print("loaded base + adapter", ADAPTER, SUBFOLDER)

## 1 · Held-out text: bits per character
Scored on the `validation` books of the corpus (never seen in training). **Bits per character** is used instead of perplexity because Llama 3 splits Tamil into near-byte pieces, which makes per-token perplexity look misleadingly low; bits per character is comparable across models and tokenizers.

In [ ]:
from datasets import load_dataset
val = load_dataset(DATASET_ID, "documents", split="validation", token=token)

ids = []
for t in val["text"]:
    ids += tok(t, add_special_tokens=True)["input_ids"] + [tok.eos_token_id]
    if len(ids) >= EVAL_TOKENS + 1:
        break
ids = ids[:EVAL_TOKENS + 1]
windows = [ids[i:i + WINDOW] for i in range(0, len(ids) - 1, WINDOW)]
windows = [w for w in windows if len(w) > 1]
n_chars = sum(len(tok.decode(w[1:])) for w in windows)   # characters of the predicted tokens
print(f"{len(windows)} windows, {sum(len(w) - 1 for w in windows):,} predicted tokens, {n_chars:,} characters")

@torch.no_grad()
def score(use_adapter):
    nll, ntok = 0.0, 0
    for w in windows:
        x = torch.tensor([w], device="cuda:0")
        if use_adapter:
            out = model(input_ids=x, labels=x)
        else:
            with model.disable_adapter():
                out = model(input_ids=x, labels=x)
        n = len(w) - 1
        nll += out.loss.item() * n
        ntok += n
    return {"bits_per_char": nll / math.log(2) / n_chars, "token_ppl": math.exp(nll / ntok),
            "nats_per_token": nll / ntok}

t0 = time.time()
held = {"base": score(False), "sltfm": score(True)}
held["bpc_reduction_pct"] = 100 * (1 - held["sltfm"]["bits_per_char"] / held["base"]["bits_per_char"])
print(f"base Llama 3 : {held['base']['bits_per_char']:.3f} bits/char  (token ppl {held['base']['token_ppl']:.2f})")
print(f"+ SLTFM      : {held['sltfm']['bits_per_char']:.3f} bits/char  (token ppl {held['sltfm']['token_ppl']:.2f})")
print(f"improvement  : {held['bpc_reduction_pct']:.1f}% fewer bits per character   [{time.time() - t0:.0f}s]")

## 2 · Example Sri Lankan Tamil questions
Three kinds: **general knowledge**, **corpus-grounded** (facts that appear in the Noolaham books, e.g. Siddha medicine and Agathiyar), and **Sri Lankan Tamil usage**. The *expected* column is a reference for the reader; judge correctness by reading.

In [ ]:
SHOT = "கேள்வி: இலங்கையின் வடக்கே உள்ள முக்கிய தமிழ் நகரம் எது?\nபதில்: யாழ்ப்பாணம்.\n\n"

QUESTIONS = [
    # (category, question, expected)
    ("General", "திருக்குறளை எழுதியவர் யார்?", "திருவள்ளுவர்"),
    ("General", "யாழ்ப்பாணம் இலங்கையின் எந்த மாகாணத்தில் உள்ளது?", "வட மாகாணம்"),
    ("General", "இலங்கையின் வணிகத் தலைநகரம் எது?", "கொழும்பு"),
    ("General", "நல்லூர் கந்தசுவாமி கோவில் எந்த நகரில் அமைந்துள்ளது?", "யாழ்ப்பாணம் (நல்லூர்)"),
    ("Corpus-grounded", "சித்த மருத்துவத்தில் கூறப்படும் மூன்று தோஷங்கள் யாவை?", "வாதம், பித்தம், கபம்"),
    ("Corpus-grounded", "திரிகடுகம் என்பது எந்த மூன்று பொருட்களின் கூட்டு?", "சுக்கு, மிளகு, திப்பிலி"),
    ("Corpus-grounded", "அகத்தியர் இயற்றியதாகக் கூறப்படும் இலக்கண நூல் எது?", "அகத்தியம்"),
    ("Corpus-grounded", "திருக்குறளில் மருத்துவம் பற்றிக் கூறும் அதிகாரத்தின் பெயர் என்ன?", "மருந்து (95ஆம் அதிகாரம்)"),
    ("Sri Lankan usage", "இலங்கைத் தமிழில் 'கதைத்தல்' என்ற சொல்லின் பொருள் என்ன?", "பேசுதல்"),
    ("Sri Lankan usage", "யாழ்ப்பாணத்தில் பனை மரத்திலிருந்து பெறப்படும் இனிப்புப் பொருள் எது?", "பனங்கட்டி / கருப்பட்டி"),
]

CONTINUATIONS = [
    "யாழ்ப்பாணம் இலங்கையின் வடக்கே அமைந்துள்ள",
    "ஈழத்துத் தமிழ் இலக்கிய வரலாற்றில்",
    "சித்த மருத்துவம் என்பது",
    "இலங்கையில் கூட்டுறவு இயக்கம்",
    "மட்டக்களப்பு மாவட்டத்தின் மக்கள்",
    "பனை மரம் யாழ்ப்பாண மக்களின் வாழ்வில்",
]

@torch.no_grad()
def generate(prompt, use_adapter, max_new_tokens, stop_newline=False):
    x = tok(prompt, return_tensors="pt").to("cuda:0")
    kw = dict(max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.15,
              pad_token_id=tok.eos_token_id)
    if use_adapter:
        y = model.generate(**x, **kw)
    else:
        with model.disable_adapter():
            y = model.generate(**x, **kw)
    text = tok.decode(y[0][x["input_ids"].shape[1]:], skip_special_tokens=True)
    if stop_newline:
        text = text.strip().split("\n")[0]
    return text.strip()

qa = []
for cat, q, exp in QUESTIONS:
    p = SHOT + f"கேள்வி: {q}\nபதில்:"
    row = {"category": cat, "question": q, "expected": exp,
           "base": generate(p, False, 120, True), "sltfm": generate(p, True, 120, True)}
    qa.append(row)
    print(f"[{cat}] {q}\n  expected: {exp}\n  base    : {row['base']}\n  SLTFM   : {row['sltfm']}\n")

## 3 · Text continuation
Each model continues the same Sri Lankan Tamil prompt (greedy decoding). Llama 3 needs ~13 tokens per Tamil word, so 300 tokens is only a few sentences.

In [ ]:
cont = []
for p in CONTINUATIONS:
    row = {"prompt": p, "base": generate(p, False, 300), "sltfm": generate(p, True, 300)}
    cont.append(row)
    print("=" * 90 + f"\nPROMPT: {p}\n\n[base Llama 3]\n{row['base']}\n\n[+ SLTFM]\n{row['sltfm']}\n")

## 4 · Save results for sharing

In [ ]:
def cell(s):
    return s.replace("|", "\\|").replace("\n", " ")

L = ["# SLTFM evaluation: base Llama 3 8B vs. SLTFM adapter (checkpoint-1100)", "",
     "## Held-out Sri Lankan Tamil text (validation books, unseen in training)", "",
     "| Model | Bits per character ↓ | Token perplexity ↓ |", "|---|---|---|",
     f"| Llama 3 8B (base) | {held['base']['bits_per_char']:.3f} | {held['base']['token_ppl']:.2f} |",
     f"| Llama 3 8B + SLTFM | **{held['sltfm']['bits_per_char']:.3f}** | **{held['sltfm']['token_ppl']:.2f}** |", "",
     f"**{held['bpc_reduction_pct']:.1f}% fewer bits per character** on {n_chars:,} held-out characters.", "",
     "## Example questions", "",
     "| Type | Question | Expected | Base Llama 3 | + SLTFM |", "|---|---|---|---|---|"]
L += [f"| {r['category']} | {cell(r['question'])} | {cell(r['expected'])} | {cell(r['base'])} | {cell(r['sltfm'])} |" for r in qa]
L += ["", "## Text continuation", ""]
for r in cont:
    L += [f"**Prompt:** {r['prompt']}", "", f"- *Base:* {cell(r['base'])}", f"- *SLTFM:* {cell(r['sltfm'])}", ""]

md_path = os.path.join(OUT_DIR, "sltfm_eval_results.md")
open(md_path, "w", encoding="utf-8").write("\n".join(L))
json.dump({"held_out": held, "questions": qa, "continuations": cont},
          open(os.path.join(OUT_DIR, "sltfm_eval_results.json"), "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
print("saved:", md_path)
print("\n".join(L[:12]))